# TCIA LDCT Data Download & HDF5 Cache Builder

Downloads the **LDCT-and-Projection-data** collection from TCIA (Mayo Clinic,
public — no account required), converts DICOM → Hounsfield-unit numpy arrays,
and writes `ldct_cache.h5` to Google Drive.

**Expected runtime:** ~30–90 min depending on how many patients you select.  
**Disk space:** ~4 GB per patient pair (low + full dose, ~300 slices each).  
Run this notebook once; `colab_quickstart.ipynb` will find the cache automatically.

---
### Structure produced
```
ldct_cache.h5
  L004_low    (num_slices, 512, 512)  float32  raw HU
  L004_full   (num_slices, 512, 512)  float32  raw HU
  L006_low    ...
  L006_full   ...
  ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tcia_utils pydicom h5py

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
DRIVE_CACHE   = '/content/drive/MyDrive/ldct_cache.h5'   # final destination
LOCAL_CACHE   = '/content/ldct_cache.h5'                 # fast scratch disk
DICOM_DIR     = '/content/ldct_dicom'                    # scratch for DICOMs

# How many patients to download (each ~4 GB; None = all available, ~50 patients)
MAX_PATIENTS  = 10

# Dose pair to cache
LOW_DOSE_TAG  = 'Low Dose Images'
FULL_DOSE_TAG = 'Full Dose Images'
BODY_PART     = 'ABDOMEN'
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
from tcia_utils import nbia
import pandas as pd

print('Querying TCIA for LDCT-and-Projection-data series...')
series  = nbia.getSeries(collection='LDCT-and-Projection-data')
df      = pd.DataFrame(series)

print(f'Total series: {len(df)}')
print('Body parts:', df['BodyPartExamined'].unique())
print('Descriptions:', df['SeriesDescription'].unique()[:10])

In [ ]:
# Keep only abdomen image series (exclude projection / sinogram data)
imgs = df[
    (df['BodyPartExamined'] == BODY_PART) &
    (df['SeriesDescription'].isin([LOW_DOSE_TAG, FULL_DOSE_TAG]))
].copy()

# Find patients that have BOTH doses
by_patient = imgs.groupby('PatientID')['SeriesDescription'].apply(set)
paired_ids = by_patient[
    by_patient.apply(lambda x: LOW_DOSE_TAG in x and FULL_DOSE_TAG in x)
].index.tolist()

print(f'Patients with paired low+full dose: {len(paired_ids)}')
print('Sample IDs:', paired_ids[:5])

if MAX_PATIENTS:
    paired_ids = paired_ids[:MAX_PATIENTS]
    print(f'Limiting to {MAX_PATIENTS} patients.')

paired_series = imgs[imgs['PatientID'].isin(paired_ids)]
print(f'Series to download: {len(paired_series)}')
paired_series[['PatientID','SeriesDescription','SeriesInstanceUID']].head(10)

In [ ]:
import os

os.makedirs(DICOM_DIR, exist_ok=True)

print(f'Downloading {len(paired_series)} series to {DICOM_DIR}...')
print('This takes a while — each series is ~1-2 GB.')

nbia.downloadSeries(
    series_data=paired_series['SeriesInstanceUID'].tolist(),
    path=DICOM_DIR,
    input_type='list'
)
print('Download complete.')

In [ ]:
# Inspect what was downloaded — tcia_utils creates one folder per SeriesInstanceUID
import glob

dcm_files = glob.glob(os.path.join(DICOM_DIR, '**', '*.dcm'), recursive=True)
print(f'Total DICOM files found: {len(dcm_files)}')

# Show folder structure
series_dirs = sorted(set(os.path.dirname(f) for f in dcm_files))
print(f'Series folders: {len(series_dirs)}')
for d in series_dirs[:6]:
    n = len(glob.glob(os.path.join(d, '*.dcm')))
    print(f'  {os.path.basename(d)[:60]}  ({n} slices)')

In [ ]:
import pydicom
import numpy as np
import h5py
from pathlib import Path

def read_series_hu(series_dir: str) -> np.ndarray:
    """Load all DICOM slices in a directory, sort by position, return HU array."""
    dcms = sorted(
        Path(series_dir).glob('*.dcm'),
        key=lambda p: float(pydicom.dcmread(str(p), stop_before_pixels=True)
                            .ImagePositionPatient[2])
    )
    if not dcms:
        raise ValueError(f'No .dcm files in {series_dir}')

    slices = []
    for p in dcms:
        ds = pydicom.dcmread(str(p))
        hu = ds.pixel_array.astype(np.float32) * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
        slices.append(hu)
    return np.stack(slices, axis=0)  # (num_slices, H, W)


# Build a lookup: SeriesInstanceUID -> (PatientID, dose_tag)
uid_meta = {
    row['SeriesInstanceUID']: (row['PatientID'], row['SeriesDescription'])
    for _, row in paired_series.iterrows()
}

# tcia_utils names folders after the SeriesInstanceUID
series_dirs = [d for d in Path(DICOM_DIR).iterdir() if d.is_dir()]

print(f'Building HDF5 cache at {LOCAL_CACHE} ...')
failed = []
with h5py.File(LOCAL_CACHE, 'w') as hf:
    for sdir in sorted(series_dirs):
        uid = sdir.name
        if uid not in uid_meta:
            continue
        pid, desc = uid_meta[uid]
        dose_key = 'low' if desc == LOW_DOSE_TAG else 'full'
        h5_key   = f'{pid}_{dose_key}'

        try:
            arr = read_series_hu(str(sdir))
            hf.create_dataset(h5_key, data=arr, compression='lzf')
            print(f'  wrote {h5_key:30s}  {arr.shape}  [{arr.min():.0f}, {arr.max():.0f}] HU')
        except Exception as e:
            print(f'  SKIP {h5_key}: {e}')
            failed.append(h5_key)

print(f'\nDone. Failed: {failed if failed else "none"}')

In [ ]:
# Verify and report what's in the cache
with h5py.File(LOCAL_CACHE, 'r') as hf:
    keys = sorted(hf.keys())
    patients = sorted({k.split('_')[0] for k in keys})
    print(f'{len(patients)} patients, {len(keys)} datasets')
    print('Patients:', patients)
    print()
    for k in keys[:6]:
        print(f'  {k:30s}  {hf[k].shape}')

In [ ]:
# Copy finished cache to Drive so it survives session restarts
import shutil

print(f'Copying {LOCAL_CACHE} -> {DRIVE_CACHE} ...')
shutil.copy(LOCAL_CACHE, DRIVE_CACHE)
print('Saved to Drive.')
print()
print('You can now train with:')
print(f'  python -m ctdenoiser.train --model ctformer --h5-cache {LOCAL_CACHE} --epochs 50 --batch-size 16')